In [1]:
"""
main.py

End-to-end orchestration: data -> features -> walk-forward models ->
backtest comparison. Run this after data_prep.py has pulled CRSP prices
and point-in-time S&P 500 membership from WRDS.

    python src/data_prep.py <wrds_username>   # one-time WRDS pull, caches to data/
    python src/main.py                        # runs the full research pipeline
"""

import pandas as pd
from pathlib import Path
from tqdm import tqdm

from features import build_feature_panel
from model import run_walk_forward, summarize_ic
from backtest import compare_models, performance_summary, compute_portfolio_returns
from visualize import plot_model_comparison


In [10]:

DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = Path.cwd().parent / "output"



In [12]:
# Forward-return horizon in months. test_months/step_months are set equal to
# HORIZON below so each walk-forward test period's realized return window is
# disjoint from the next -- required for backtest.py to validly compound
# them as a sequential return series. See model.run_walk_forward's docstring.
HORIZON = 3

print("Loading CRSP price panel and point-in-time membership...")
daily_panel = pd.read_parquet(DATA_DIR / "prices_wrds.parquet")
membership = pd.read_parquet(DATA_DIR / "sp500_membership.parquet")


Loading CRSP price panel and point-in-time membership...


In [ ]:
daily_panel
membership

,permno,date,prc,ret,vol,shrout,cfacpr,cfacshr,mkt_cap,dollar_vol
0,10104,2011-01-03,31.62,0.010224,21136353.0,5052420.0,1.0,1.0,1.597575e+11,6.683315e+08
1,10104,2011-01-04,31.48,-0.004428,22978313.0,5052420.0,1.0,1.0,1.590502e+11,7.233573e+08
2,10104,2011-01-05,31.04,-0.013977,36464087.0,5052420.0,1.0,1.0,1.568271e+11,1.131845e+09
3,10104,2011-01-06,31.17,0.004188,21963429.0,5052420.0,1.0,1.0,1.574839e+11,6.846001e+08
4,10104,2011-01-07,31.03,-0.004491,27819266.0,5052420.0,1.0,1.0,1.567766e+11,8.632318e+08
...,...,...,...,...,...,...,...,...,...,...
1928342,93436,2026-03-25,385.95,0.007623,54662141.0,3752432.0,1.0,1.0,1.448251e+12,2.109685e+10
1928343,93436,2026-03-26,372.11,-0.035860,55069227.0,3752432.0,1.0,1.0,1.396317e+12,2.049181e+10
1928344,93436,2026-03-27,361.83,-0.027626,61316932.0,3752432.0,1.0,1.0,1.357742e+12,2.218631e+10
1928345,93436,2026-03-30,355.28,-0.018102,67433860.0,3752432.0,1.0,1.0,1.333164e+12,2.395790e+10


In [ ]:

print("Building feature panel...")
feature_panel = build_feature_panel(daily_panel, membership, horizon=HORIZON)
print(f"Feature panel shape: {feature_panel.shape}")
print(f"Date range: {feature_panel['date'].min()} to {feature_panel['date'].max()}")
print(
    f"Unique PERMNOs represented (point-in-time members only): "
    f"{feature_panel['permno'].nunique()}"
)

OUTPUT_DIR.mkdir(exist_ok=True)
feature_panel.to_parquet(OUTPUT_DIR / "feature_panel.parquet", index=False)

predictions_by_model = {}

for model_type in ["linear", "gbm"]:
    print(f"\nRunning walk-forward for model: {model_type}")
    preds = run_walk_forward(
        feature_panel, model_type=model_type,
        horizon=HORIZON, test_months=1, step_months=HORIZON,
    )
    predictions_by_model[model_type] = preds

    print(f"-- {model_type} IC summary --")
    summarize_ic(preds)

    preds.to_parquet(OUTPUT_DIR / f"predictions_{model_type}.parquet", index=False)

print("\n--- Model comparison ---")
comparison = compare_models(predictions_by_model, freq=12 // HORIZON)
print(comparison)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv")
plot_model_comparison(OUTPUT_DIR / "model_comparison.csv", OUTPUT_DIR / "model_comparison.png")
'''